# Extract Window Features From Raw PCAPs

This notebook starts from the raw `eth1-*` PCAP files and creates the window-based feature CSVs used by the classification and defense notebooks.

Output files are saved in `extracted_data/` as `features_YYYY_MM_DD.csv`.

In [1]:
from pathlib import Path

import pandas as pd

from cross_day_device_classification import (
    WINDOW_SIZE,
    extract_day_features,
    load_device_mapping,
)

RAW_MONTH_DIR = Path("2018/03")
OUTPUT_DIR = Path("extracted_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to True if you want to overwrite existing files in extracted_data.
FORCE_REEXTRACT = False

print(f"Raw month directory: {RAW_MONTH_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


Raw month directory: /home/leiy28/CS203-Project/2018/03
Output directory: /home/leiy28/CS203-Project/extracted_data


## Discover Raw Day Folders

The notebook extracts every day folder under `2018/03` that contains at least one `eth1-*` raw capture file.

In [2]:
if not RAW_MONTH_DIR.exists():
    raise FileNotFoundError(f"Missing raw data directory: {RAW_MONTH_DIR}")

day_dirs = []
for day_dir in sorted(RAW_MONTH_DIR.iterdir()):
    if not day_dir.is_dir():
        continue
    pcap_count = len(list(day_dir.glob("eth1-*")))
    if pcap_count:
        day_dirs.append((day_dir, pcap_count))

if not day_dirs:
    raise FileNotFoundError(f"No eth1-* raw captures found under {RAW_MONTH_DIR}")

pd.DataFrame(
    [
        {
            "day": f"2018-03-{day_dir.name}",
            "raw_dir": str(day_dir),
            "pcap_files": pcap_count,
        }
        for day_dir, pcap_count in day_dirs
    ]
)


,day,raw_dir,pcap_files
0,2018-03-20,2018/03/20,288
1,2018-03-21,2018/03/21,288


## Load Device Mapping

The mapping connects local IP addresses to device names. Packets whose source or destination IP is not in this mapping are ignored.

In [3]:
ip_to_device = load_device_mapping("device_mapping.csv")
print(f"Loaded {len(ip_to_device)} mapped device IPs")

mapping_preview = pd.DataFrame(
    [{"ip": ip, "device": device} for ip, device in sorted(ip_to_device.items())]
)
mapping_preview.head(10)


Loaded 66 mapped device IPs


,ip,device
0,192.168.0.1,Gateway
1,192.168.0.10,NestCamera
2,192.168.0.113,UbuntuDesktop
3,192.168.0.12,BelkinWeMoMotionSensor
4,192.168.0.13,LIFXVirtualBulb
5,192.168.0.138,AndroidTablet
6,192.168.0.14,BelkinWeMoSwitch
7,192.168.0.15,AmazonEchoGen1
8,192.168.0.151,iPhone
9,192.168.0.159,iPad


## Extract And Save Features

For each device in each fixed time window, the extractor saves packet counts, byte counts, direction-specific counts, and packet-size summaries.

In [4]:
extracted = {}
summary_rows = []

for day_dir, pcap_count in day_dirs:
    day = f"2018-03-{day_dir.name}"
    output_path = OUTPUT_DIR / f"features_{day.replace('-', '_')}.csv"

    if FORCE_REEXTRACT and output_path.exists():
        output_path.unlink()

    print(f"\nExtracting {day} from {pcap_count} raw PCAP files")
    feature_df = extract_day_features(
        day_dir,
        ip_to_device,
        cache_path=output_path,
        window_size=WINDOW_SIZE,
    )
    extracted[day] = feature_df

    summary_rows.append(
        {
            "day": day,
            "raw_dir": str(day_dir),
            "output_file": str(output_path),
            "pcap_files": pcap_count,
            "windows": len(feature_df),
            "devices": feature_df["label"].nunique(),
            "window_size_seconds": WINDOW_SIZE,
        }
    )

summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "extraction_summary.csv"
summary.to_csv(summary_path, index=False)
summary



Extracting 2018-03-20 from 288 raw PCAP files
Extracting 2018/03/20 (1/288)
Extracting 2018/03/20 (48/288)
Extracting 2018/03/20 (96/288)
Extracting 2018/03/20 (144/288)
Extracting 2018/03/20 (192/288)
Extracting 2018/03/20 (240/288)
Extracting 2018/03/20 (288/288)

Extracting 2018-03-21 from 288 raw PCAP files
Extracting 2018/03/21 (1/288)
Extracting 2018/03/21 (48/288)
Extracting 2018/03/21 (96/288)
Extracting 2018/03/21 (144/288)
Extracting 2018/03/21 (192/288)
Extracting 2018/03/21 (240/288)
Extracting 2018/03/21 (288/288)


,day,raw_dir,output_file,pcap_files,windows,devices,window_size_seconds
0,2018-03-20,2018/03/20,extracted_data/features_2018_03_20.csv,288,257070,45,5
1,2018-03-21,2018/03/21,extracted_data/features_2018_03_21.csv,288,293303,49,5


## Sanity Checks

In [5]:
for day, feature_df in extracted.items():
    print(f"\n{day}")
    print(feature_df.head())
    print(feature_df["label"].value_counts().head(10))

print(f"\nSaved extracted files to: {OUTPUT_DIR.resolve()}")



2018-03-20
    window_start              device  pkt_count  total_bytes  outgoing_pkts  \
11           0.0      AmazonEchoGen1          6          606              4   
10           0.0             Gateway          2          168              2   
9            0.0      GoogleHomeMini         22         1728             13   
3            0.0  HarmonKardonInvoke          9         2308              5   
0            0.0  LogitechLogiCircle        232       183500            145   

    incoming_pkts  outgoing_bytes  incoming_bytes  avg_pkt_size  max_pkt_size  \
11              2             485             121    101.000000           187   
10              0             168               0     84.000000            84   
9               9             912             816     78.545455           156   
3               4            1584             724    256.444444          1289   
0              87          178976            4524    790.948276          1500   

    min_pkt_size          